# 02 - Exploratory Data Analysis (EDA)

This notebook performs comprehensive exploratory data analysis to understand patterns, relationships, and insights from the Telco Customer Churn dataset.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Set style for visualizations
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Create directories for saving figures
FIGURES_PATH = '../reports/figures/'
os.makedirs(FIGURES_PATH, exist_ok=True)

## 2. Load Data

In [ ]:
# Load the data from processed folder
df = pd.read_csv('../data/processed/raw_data_loaded.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nDataset loaded successfully!")

## 3. Target Variable Analysis

In [ ]:
# Churn distribution
plt.figure(figsize=(8, 6))
churn_counts = df['Churn'].value_counts()
plt.pie(churn_counts, labels=churn_counts.index, autopct='%1.1f%%', startangle=90, colors=['#66c2a5', '#fc8d62'])
plt.title('Churn Distribution')
plt.savefig(FIGURES_PATH + 'churn_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Churn Rate: {(churn_counts['Yes'] / len(df) * 100):.2f}%")

## 4. Numerical Features Analysis

In [ ]:
# Identify numerical columns
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print("Numerical Columns:", numerical_cols)

In [ ]:
# Distribution of numerical features
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for idx, col in enumerate(numerical_cols):
    if idx < len(axes):
        sns.histplot(df[col], kde=True, ax=axes[idx], color='skyblue')
        axes[idx].set_title(f'Distribution of {col}')
        axes[idx].set_xlabel(col)
        axes[idx].set_ylabel('Frequency')

plt.tight_layout()
plt.savefig(FIGURES_PATH + 'numerical_distributions.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Box plots for numerical features by Churn
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for idx, col in enumerate(numerical_cols):
    if idx < len(axes):
        sns.boxplot(x='Churn', y=col, data=df, ax=axes[idx], palette='Set2')
        axes[idx].set_title(f'{col} by Churn Status')

plt.tight_layout()
plt.savefig(FIGURES_PATH + 'numerical_by_churn.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. Categorical Features Analysis

In [ ]:
# Identify categorical columns (excluding CustomerID)
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
categorical_cols = [col for col in categorical_cols if col != 'CustomerID']
print("Categorical Columns:", categorical_cols)

In [ ]:
# Categorical features distribution
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for idx, col in enumerate(categorical_cols):
    if idx < len(axes):
        sns.countplot(x=col, data=df, ax=axes[idx], palette='Set3')
        axes[idx].set_title(f'Distribution of {col}')
        axes[idx].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(FIGURES_PATH + 'categorical_distributions.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Categorical features by Churn
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for idx, col in enumerate(categorical_cols):
    if idx < len(axes):
        sns.countplot(x=col, hue='Churn', data=df, ax=axes[idx], palette='Set2')
        axes[idx].set_title(f'{col} by Churn Status')
        axes[idx].tick_params(axis='x', rotation=45)
        axes[idx].legend(title='Churn')

plt.tight_layout()
plt.savefig(FIGURES_PATH + 'categorical_by_churn.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Correlation Analysis

In [ ]:
# Correlation matrix for numerical features
correlation_matrix = df[numerical_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Correlation Matrix of Numerical Features')
plt.savefig(FIGURES_PATH + 'correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nCorrelation with Churn (encoded):")
df_encoded = df.copy()
df_encoded['Churn_Encoded'] = df_encoded['Churn'].map({'Yes': 1, 'No': 0})
print(df_encoded[numerical_cols + ['Churn_Encoded']].corr()['Churn_Encoded'].sort_values(ascending=False))

## 7. Age Analysis

In [ ]:
# Age distribution by Churn
plt.figure(figsize=(12, 6))
sns.histplot(data=df, x='Age', hue='Churn', kde=True, bins=30, palette='Set2')
plt.title('Age Distribution by Churn Status')
plt.xlabel('Age')
plt.ylabel('Frequency')
plt.savefig(FIGURES_PATH + 'age_by_churn.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Age groups analysis
df['Age_Group'] = pd.cut(df['Age'], bins=[0, 30, 40, 50, 60, 100], labels=['<30', '30-40', '40-50', '50-60', '60+'])

plt.figure(figsize=(10, 6))
sns.countplot(x='Age_Group', hue='Churn', data=df, palette='Set2')
plt.title('Churn by Age Group')
plt.xlabel('Age Group')
plt.ylabel('Count')
plt.savefig(FIGURES_PATH + 'churn_by_age_group.png', dpi=300, bbox_inches='tight')
plt.show()

# Calculate churn rate by age group
age_churn_rate = df.groupby('Age_Group')['Churn'].apply(lambda x: (x == 'Yes').mean() * 100)
print("\nChurn Rate by Age Group:")
print(age_churn_rate)

## 8. Tenure Analysis

In [ ]:
# Tenure distribution by Churn
plt.figure(figsize=(12, 6))
sns.histplot(data=df, x='Tenure', hue='Churn', kde=True, bins=30, palette='Set2')
plt.title('Tenure Distribution by Churn Status')
plt.xlabel('Tenure (months)')
plt.ylabel('Frequency')
plt.savefig(FIGURES_PATH + 'tenure_by_churn.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Tenure groups analysis
df['Tenure_Group'] = pd.cut(df['Tenure'], bins=[0, 12, 24, 36, 48, 60, 100], 
                            labels=['0-12', '12-24', '24-36', '36-48', '48-60', '60+'])

plt.figure(figsize=(12, 6))
sns.countplot(x='Tenure_Group', hue='Churn', data=df, palette='Set2')
plt.title('Churn by Tenure Group')
plt.xlabel('Tenure Group (months)')
plt.ylabel('Count')
plt.savefig(FIGURES_PATH + 'churn_by_tenure_group.png', dpi=300, bbox_inches='tight')
plt.show()

# Calculate churn rate by tenure group
tenure_churn_rate = df.groupby('Tenure_Group')['Churn'].apply(lambda x: (x == 'Yes').mean() * 100)
print("\nChurn Rate by Tenure Group:")
print(tenure_churn_rate)

## 9. Charges Analysis

In [ ]:
# Monthly Charges by Churn
plt.figure(figsize=(12, 6))
sns.boxplot(x='Churn', y='MonthlyCharges', data=df, palette='Set2')
plt.title('Monthly Charges by Churn Status')
plt.xlabel('Churn')
plt.ylabel('Monthly Charges')
plt.savefig(FIGURES_PATH + 'monthly_charges_by_churn.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Total Charges by Churn
plt.figure(figsize=(12, 6))
sns.boxplot(x='Churn', y='TotalCharges', data=df, palette='Set2')
plt.title('Total Charges by Churn Status')
plt.xlabel('Churn')
plt.ylabel('Total Charges')
plt.savefig(FIGURES_PATH + 'total_charges_by_churn.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Average charges by Churn
print("\nAverage Monthly Charges by Churn:")
print(df.groupby('Churn')['MonthlyCharges'].mean())

print("\nAverage Total Charges by Churn:")
print(df.groupby('Churn')['TotalCharges'].mean())

## 10. Contract and Payment Method Analysis

In [ ]:
# Churn rate by Contract type
contract_churn = df.groupby('Contract')['Churn'].value_counts(normalize=True).unstack()
contract_churn_rate = contract_churn['Yes'] * 100

plt.figure(figsize=(10, 6))
contract_churn_rate.plot(kind='bar', color='coral')
plt.title('Churn Rate by Contract Type')
plt.xlabel('Contract Type')
plt.ylabel('Churn Rate (%)')
plt.xticks(rotation=45)
plt.savefig(FIGURES_PATH + 'churn_rate_by_contract.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nChurn Rate by Contract Type:")
print(contract_churn_rate)

In [ ]:
# Churn rate by Payment Method
payment_churn = df.groupby('PaymentMethod')['Churn'].value_counts(normalize=True).unstack()
payment_churn_rate = payment_churn['Yes'] * 100

plt.figure(figsize=(10, 6))
payment_churn_rate.plot(kind='bar', color='lightblue')
plt.title('Churn Rate by Payment Method')
plt.xlabel('Payment Method')
plt.ylabel('Churn Rate (%)')
plt.xticks(rotation=45)
plt.savefig(FIGURES_PATH + 'churn_rate_by_payment.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nChurn Rate by Payment Method:")
print(payment_churn_rate)

## 11. Gender Analysis

In [ ]:
# Churn rate by Gender
gender_churn = df.groupby('Gender')['Churn'].value_counts(normalize=True).unstack()
gender_churn_rate = gender_churn['Yes'] * 100

plt.figure(figsize=(8, 6))
gender_churn_rate.plot(kind='bar', color=['lightgreen', 'lightpink'])
plt.title('Churn Rate by Gender')
plt.xlabel('Gender')
plt.ylabel('Churn Rate (%)')
plt.xticks(rotation=0)
plt.savefig(FIGURES_PATH + 'churn_rate_by_gender.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nChurn Rate by Gender:")
print(gender_churn_rate)

## 12. Key Insights Summary

In [ ]:
print("="*60)
print("KEY INSIGHTS FROM EDA")
print("="*60)

print(f"\n1. Overall Churn Rate: {(df['Churn'] == 'Yes').mean() * 100:.2f}%")

print(f"\n2. Average Age of Churned Customers: {df[df['Churn'] == 'Yes']['Age'].mean():.2f}")
print(f"   Average Age of Non-Churned Customers: {df[df['Churn'] == 'No']['Age'].mean():.2f}")

print(f"\n3. Average Tenure of Churned Customers: {df[df['Churn'] == 'Yes']['Tenure'].mean():.2f} months")
print(f"   Average Tenure of Non-Churned Customers: {df[df['Churn'] == 'No']['Tenure'].mean():.2f} months")

print(f"\n4. Average Monthly Charges of Churned Customers: ${df[df['Churn'] == 'Yes']['MonthlyCharges'].mean():.2f}")
print(f"   Average Monthly Charges of Non-Churned Customers: ${df[df['Churn'] == 'No']['MonthlyCharges'].mean():.2f}")

print(f"\n5. Contract Type with Highest Churn Rate:")
print(f"   {contract_churn_rate.idxmax()} ({contract_churn_rate.max():.2f}%)")

print(f"\n6. Payment Method with Highest Churn Rate:")
print(f"   {payment_churn_rate.idxmax()} ({payment_churn_rate.max():.2f}%)")

print("="*60)

## 13. Save Processed Data for Feature Engineering

In [ ]:
# Save the dataframe with engineered groups for next steps
df.to_csv('../data/processed/eda_completed.csv', index=False)
print("\nData saved to ../data/processed/eda_completed.csv")
print("EDA completed successfully!")

## Summary

This notebook completed comprehensive EDA:
- Analyzed target variable distribution (churn rate)
- Examined numerical features distributions and relationships
- Analyzed categorical features and their impact on churn
- Investigated correlations between features
- Performed detailed analysis on Age, Tenure, and Charges
- Examined Contract and Payment Method impact on churn
- Generated key business insights
- Saved processed data for feature engineering phase